# 第 13 节：Baseline 与 Advantage

## 位置
REINFORCE (12) → **Baseline 与 Advantage (13)** → Actor-Critic (14) → A2C (15)

## 学习目标
1. 理解为什么 REINFORCE 有高方差
2. 掌握 Baseline 的数学原理：为什么引入 baseline 不改变期望（无偏性）
3. 理解 Advantage A(s,a) = Q(s,a) - V(s) 的含义
4. 从数学上证明 baseline 不引入偏差
5. 用数值实验验证 baseline 的无偏性和方差降低效果
6. 对比 REINFORCE 有无 baseline 的效果差异
7. 理解 V(s) 作为最优 baseline 的直觉
8. 建立从 baseline 到 Actor-Critic 的连接

## 1. 为什么 REINFORCE 有高方差？

### 回顾 REINFORCE

REINFORCE 的策略梯度估计为：

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t | s_t) \cdot G_t \right]$$

其中 $G_t = \sum_{k=t}^{T} \gamma^{k-t} r_k$ 是蒙特卡洛回报。

### 方差来源

$G_t$ 是一个随机变量，它**累积了整个未来轨迹的所有随机性**：

1. **策略随机性**：未来每一步的动作都是随机采样的
2. **环境随机性**：环境转移和奖励都可能随机
3. **时间累积**：步数越多，$G_t$ 的方差越大

因此，不同 episode 之间的 $G_t$ 可能差异巨大，导致梯度更新方向不稳定。

### 数学表达

$$\text{Var}[\nabla_\theta J(\theta)] \propto \text{Var}[G_t]$$

$G_t$ 的方差随 episode 长度线性增长，这就是 REINFORCE 高方差的根源。

## 2. Baseline 的核心思想

### 直觉

如果我们从 $G_t$ 中减去一个与动作无关的**基线函数** $b(s_t)$，期望不变，但方差可以减小：

$$\nabla_\theta J(\theta) = \mathbb{E} \left[ \nabla_\theta \log \pi_\theta(a_t | s_t) \cdot (G_t - b(s_t)) \right]$$

### 关键要求

- $b(s_t)$ 可以依赖于当前状态 $s_t$
- $b(s_t)$ **不能**依赖于动作 $a_t$
- $b(s_t)$ 可以是**任何函数**（甚至可以是常数）

### Advantage 定义

当我们把 baseline 设为状态价值函数 $V(s_t)$ 时，得到 **优势函数 (Advantage Function)**：

$$A(s_t, a_t) = Q(s_t, a_t) - V(s_t)$$

> **重要**：在 REINFORCE with baseline 中，我们实际上使用 $A_t = G_t - V(s_t)$ 作为优势估计。这不是真正的 Q 函数，而是用 MC 回报 $G_t$ 代替 $Q(s_t, a_t)$ 的**无偏估计**。

## 3. 数学证明：Baseline 不引入偏差

我们需要证明加入 baseline b(s) 后策略梯度的期望不变：

$$\mathbb{E}_{a \sim \pi} \left[ \nabla_\theta \log \pi_\theta(a|s) \cdot b(s) \right] = 0$$

### 逐步推导

**Step 1**: 写出期望的显式形式

$$\mathbb{E}_{a \sim \pi} \left[ \nabla_\theta \log \pi_\theta(a|s) \cdot b(s) \right]
= \sum_a \pi_\theta(a|s) \cdot \nabla_\theta \log \pi_\theta(a|s) \cdot b(s)$$

**Step 2**: 将 b(s) 提到求和外（因为它不依赖 a）

$$= b(s) \cdot \sum_a \pi_\theta(a|s) \cdot \nabla_\theta \log \pi_\theta(a|s)$$

**Step 3**: 使用对数导数技巧 (log-derivative trick)

回忆：$\nabla_\theta \log \pi_\theta = \frac{\nabla_\theta \pi_\theta}{\pi_\theta}$

$$\sum_a \pi_\theta \cdot \nabla_\theta \log \pi_\theta
= \sum_a \pi_\theta \cdot \frac{\nabla_\theta \pi_\theta}{\pi_\theta}
= \sum_a \nabla_\theta \pi_\theta$$

**Step 4**: 梯度交换求和顺序

$$\sum_a \nabla_\theta \pi_\theta(a|s)
= \nabla_\theta \sum_a \pi_\theta(a|s)$$

**Step 5**: 概率和为 1

$$\sum_a \pi_\theta(a|s) = 1 \quad \Rightarrow \quad \nabla_\theta \sum_a \pi_\theta(a|s) = \nabla_\theta 1 = 0$$

**Step 6**: 结论

$$\mathbb{E}_{a \sim \pi} \left[ \nabla_\theta \log \pi_\theta(a|s) \cdot b(s) \right] = b(s) \cdot 0 = 0$$

### 含义

- Baseline 不改变策略梯度的**期望值**（无偏）
- Baseline 可以改变策略梯度的**方差**
- 一个好的 baseline 能显著降低方差，加速收敛

In [ ]:
# Cell 5: 数值验证 Baseline 的无偏性

import sys; sys.path.insert(0, '/workspace/data/vggt-omega/rl')
from rl_course.utils.seeding import set_seed; set_seed(42)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# 构造一个简单的 3-状态 MDP 来验证
set_seed(42)

# 简单策略分布 (3个状态, 2个动作)
theta = torch.tensor([[0.5, -0.3], [1.0, 0.2], [-0.8, 0.6]], requires_grad=True)

# 定义 baseline 函数 b(s) — 用任意状态相关的值
baselines = torch.tensor([10.0, -5.0, 3.0])  # 任意大的数，强调效果

num_samples = 50000
estimates_with_baseline = []
estimates_without_baseline = []

for _ in range(100):  # 重复 100 次估计
    grad_est_wo = []
    grad_est_w = []

    for s in range(3):
        logits_s = theta[s]
        probs = F.softmax(logits_s, dim=0)

        # 采样大量动作
        actions = torch.multinomial(probs, num_samples, replacement=True)

        # 计算 log prob
        log_probs = F.log_softmax(logits_s, dim=0)
        sampled_log_probs = log_probs[actions]

        # G_t 用随机值模拟（保证有方差）
        G_t = torch.randn(num_samples) * 2 + 1.0

        # 无 baseline 的梯度估计
        grad_wo = (sampled_log_probs * G_t).mean()
        grad_est_wo.append(grad_wo.item())

        # 有 baseline 的梯度估计
        grad_w = (sampled_log_probs * (G_t - baselines[s])).mean()
        grad_est_w.append(grad_w.item())

    estimates_without_baseline.append(grad_est_wo)
    estimates_with_baseline.append(grad_est_w)

estimates_wo = np.array(estimates_without_baseline)
estimates_w = np.array(estimates_with_baseline)

print("===== 数值验证结果 =====")
print(f"{'状态':<6} {'无 baseline 均值':<16} {'有 baseline 均值':<16} {'差异':<10}")
print("-" * 54)
for s in range(3):
    diff = estimates_wo[:, s].mean() - estimates_w[:, s].mean()
    print(f"s{s+1:<4} {estimates_wo[:, s].mean():<16.6f} {estimates_w[:, s].mean():<16.6f} {diff:<10.6f}")

print("\n结论：有无 baseline 的梯度估计均值几乎相同 (差异约等于 0)，验证了无偏性。")

# 验证 baseline 降低方差
print(f"\n===== 方差对比 =====")
for s in range(3):
    var_wo = np.var(estimates_wo[:, s])
    var_w = np.var(estimates_w[:, s])
    ratio = var_wo / (var_w + 1e-10)
    print(f"状态 s{s+1}: 无 baseline Var={var_wo:.6f}, 有 baseline Var={var_w:.6f}, 比率={ratio:.2f}x")


## 4. Advantage 作为方差降低技术

### 最优 Baseline

**最优 baseline** 是使方差最小的 baseline。可以证明，最优 baseline 是价值加权平均：

$$b^*(s) = \frac{\mathbb{E}_{a \sim \pi} \left[ (\nabla_\theta \log \pi)^2 \cdot Q(s,a) \right]}{\mathbb{E}_{a \sim \pi} \left[ (\nabla_\theta \log \pi)^2 \right]}$$

在实践中，我们通常取 $b(s) = V(s)$，因为：

1. $V(s) = \mathbb{E}_a[Q(s,a)]$，与最优 baseline 接近（在梯度幅度随动作变化不大时）
2. $V(s)$ 本身可以用监督学习轻松学习
3. 有清晰的理论解释：$A(s,a) = Q(s,a) - V(s)$ 表示动作的相对好坏

### 方差降低的直觉

| 组件 | 方差 | 说明 |
|------|------|------|
| $G_t$ | 高 | 累积所有随机性 |
| $G_t - V(s_t)$ | 中 | 减去已知的期望回报 |
| $A(s,a) = Q - V$ | 低 | 只关注动作带来的差异 |

### Advantage 的意义

- **正 Advantage**：该动作优于平均水平 → 增加其概率
- **负 Advantage**：该动作劣于平均水平 → 减少其概率
- Advantage 的尺度比 $G_t$ 小得多 → 梯度更新更稳定

In [ ]:
# Cell 7: 环境准备 — 对比 REINFORCE 有/无 baseline

import sys; sys.path.insert(0, '/workspace/data/vggt-omega/rl')
from rl_course.utils.seeding import set_seed; set_seed(42)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import gymnasium as gym
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

# 配置 matplotlib
%matplotlib inline
plt.rcParams.update({'figure.dpi': 100, 'savefig.dpi': 100})

FIG_DIR = "outputs/figures"
os.makedirs(FIG_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# CartPole 环境
env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
n_actions = env.action_space.n
print(f"CartPole: state_dim={state_dim}, n_actions={n_actions}")
env.close()


In [ ]:
# Cell 8: 从零实现 REINFORCE (无 baseline)

class REINFORCEVanilla:
    """REINFORCE without baseline"""

    def __init__(self, state_dim, n_actions, hidden_dim=128, lr=1e-3, gamma=0.99):
        self.gamma = gamma
        self.policy = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_actions),
        ).to(device)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=lr)
        self.reset_buffer()

    def reset_buffer(self):
        self.states, self.actions, self.rewards, self.log_probs = [], [], [], []

    def act(self, state, train=True):
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        if train:
            logits = self.policy(state_t)
            probs = F.softmax(logits, dim=-1)
            log_probs = F.log_softmax(logits, dim=-1)
            action = torch.multinomial(probs, 1).squeeze()
            log_prob = log_probs[0, action]
            self.states.append(state_t.squeeze(0))
            self.actions.append(action.item())
            self.log_probs.append(log_prob)
            return action.item()
        else:
            with torch.no_grad():
                logits = self.policy(state_t)
                action = torch.argmax(logits, dim=-1).item()
            return action

    def update(self):
        if len(self.states) == 0:
            return {"policy_loss": 0.0, "episode_return": 0.0, "grad_norm": 0.0}
        rewards = torch.FloatTensor(self.rewards).to(device)
        log_probs = torch.stack(self.log_probs).to(device)
        returns = []
        G = 0.0
        for r in reversed(rewards):
            G = r + self.gamma * G
            returns.insert(0, G)
        returns = torch.FloatTensor(returns).to(device)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        policy_loss = -(log_probs * returns).mean()
        self.optimizer.zero_grad()
        policy_loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(self.policy.parameters(), 10.0)
        self.optimizer.step()
        episode_return = rewards.sum().item()
        loss_val = policy_loss.item()
        self.reset_buffer()
        return {"policy_loss": loss_val, "episode_return": episode_return, "grad_norm": grad_norm.item()}


In [ ]:
# Cell 9: 从零实现 REINFORCE with Baseline

class REINFORCEWithBaseline:
    """REINFORCE with learned value baseline"""

    def __init__(self, state_dim, n_actions, hidden_dim=128, lr=1e-3, gamma=0.99):
        self.gamma = gamma
        self.policy = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_actions),
        ).to(device)
        self.value = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        ).to(device)
        self.optimizer = optim.Adam(
            list(self.policy.parameters()) + list(self.value.parameters()), lr=lr
        )
        self.reset_buffer()

    def reset_buffer(self):
        self.states, self.actions, self.rewards, self.log_probs = [], [], [], []

    def act(self, state, train=True):
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        if train:
            logits = self.policy(state_t)
            probs = F.softmax(logits, dim=-1)
            log_probs = F.log_softmax(logits, dim=-1)
            action = torch.multinomial(probs, 1).squeeze()
            log_prob = log_probs[0, action]
            self.states.append(state_t.squeeze(0))
            self.actions.append(action.item())
            self.log_probs.append(log_prob)
            return action.item()
        else:
            with torch.no_grad():
                logits = self.policy(state_t)
                action = torch.argmax(logits, dim=-1).item()
            return action

    def update(self):
        if len(self.states) == 0:
            return {"policy_loss": 0.0, "value_loss": 0.0, "episode_return": 0.0, "grad_norm": 0.0}
        states = torch.stack(self.states).to(device)
        rewards = torch.FloatTensor(self.rewards).to(device)
        log_probs = torch.stack(self.log_probs).to(device)
        returns = []
        G = 0.0
        for r in reversed(rewards):
            G = r + self.gamma * G
            returns.insert(0, G)
        returns = torch.FloatTensor(returns).to(device)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        values = self.value(states).squeeze(-1)
        advantages = returns - values
        policy_loss = -(log_probs * advantages.detach()).mean()
        value_loss = F.mse_loss(values, returns)
        total_loss = policy_loss + value_loss
        self.optimizer.zero_grad()
        total_loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(
            list(self.policy.parameters()) + list(self.value.parameters()), 10.0
        )
        self.optimizer.step()
        episode_return = rewards.sum().item()
        self.reset_buffer()
        return {
            "policy_loss": policy_loss.item(),
            "value_loss": value_loss.item(),
            "episode_return": episode_return,
            "grad_norm": grad_norm.item(),
        }


In [ ]:
# Cell 10: 训练两个智能体并对比

N_EPISODES = 300
GAMMA = 0.99
LR = 5e-4

# 创建智能体
agent_vanilla = REINFORCEVanilla(state_dim, n_actions, lr=LR, gamma=GAMMA)
agent_baseline = REINFORCEWithBaseline(state_dim, n_actions, lr=LR, gamma=GAMMA)

# 记录指标
metrics_vanilla = {"returns": [], "losses": [], "grad_norms": []}
metrics_baseline = {"returns": [], "policy_losses": [], "value_losses": [], "grad_norms": []}

set_seed(42)

print("训练无 baseline REINFORCE...")
for ep in tqdm(range(N_EPISODES)):
    env = gym.make("CartPole-v1")
    state, _ = env.reset()
    agent_vanilla.reset_buffer()
    done = False
    while not done:
        action = agent_vanilla.act(state, train=True)
        next_state, reward, terminated, truncated, _ = env.step(action)
        agent_vanilla.rewards.append(reward)
        done = terminated or truncated
        state = next_state
    info = agent_vanilla.update()
    metrics_vanilla["returns"].append(info["episode_return"])
    metrics_vanilla["losses"].append(info["policy_loss"])
    metrics_vanilla["grad_norms"].append(info["grad_norm"])
    env.close()

print("\\n训练带 baseline REINFORCE...")
for ep in tqdm(range(N_EPISODES)):
    env = gym.make("CartPole-v1")
    state, _ = env.reset()
    agent_baseline.reset_buffer()
    done = False
    while not done:
        action = agent_baseline.act(state, train=True)
        next_state, reward, terminated, truncated, _ = env.step(action)
        agent_baseline.rewards.append(reward)
        done = terminated or truncated
        state = next_state
    info = agent_baseline.update()
    metrics_baseline["returns"].append(info["episode_return"])
    metrics_baseline["policy_losses"].append(info["policy_loss"])
    metrics_baseline["value_losses"].append(info["value_loss"])
    metrics_baseline["grad_norms"].append(info["grad_norm"])
    env.close()

print("\\n训练完成!")


In [ ]:
# Cell 11: 对比两种方法的训练曲线

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

# 滑动平均
def smooth(data, window=10):
    if len(data) < window:
        return data
    return np.convolve(data, np.ones(window)/window, mode='valid')

window = 10
FIG_DIR = "outputs/figures"

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 左上：Returns
axes[0, 0].plot(metrics_vanilla["returns"], alpha=0.3, linewidth=0.5, label='Vanilla', color='steelblue')
axes[0, 0].plot(metrics_baseline["returns"], alpha=0.3, linewidth=0.5, label='With Baseline', color='coral')
axes[0, 0].plot(range(window-1, N_EPISODES), smooth(metrics_vanilla["returns"], window),
                linewidth=2, color='steelblue', label=f'Vanilla (smoothed)')
axes[0, 0].plot(range(window-1, N_EPISODES), smooth(metrics_baseline["returns"], window),
                linewidth=2, color='coral', label=f'With Baseline (smoothed)')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Return')
axes[0, 0].set_title('Episode Returns')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 右上：Policy Loss
axes[0, 1].plot(metrics_vanilla["losses"], alpha=0.3, linewidth=0.5, color='steelblue')
axes[0, 1].plot(metrics_baseline["policy_losses"], alpha=0.3, linewidth=0.5, color='coral')
axes[0, 1].plot(range(window-1, N_EPISODES), smooth(metrics_vanilla["losses"], window),
                linewidth=2, color='steelblue', label='Vanilla')
axes[0, 1].plot(range(window-1, N_EPISODES), smooth(metrics_baseline["policy_losses"], window),
                linewidth=2, color='coral', label='With Baseline')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Policy Loss')
axes[0, 1].set_title('Policy Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 左下：Gradient Norm
axes[1, 0].plot(metrics_vanilla["grad_norms"], alpha=0.3, linewidth=0.5, color='steelblue')
axes[1, 0].plot(metrics_baseline["grad_norms"], alpha=0.3, linewidth=0.5, color='coral')
axes[1, 0].plot(range(window-1, N_EPISODES), smooth(metrics_vanilla["grad_norms"], window),
                linewidth=2, color='steelblue', label='Vanilla')
axes[1, 0].plot(range(window-1, N_EPISODES), smooth(metrics_baseline["grad_norms"], window),
                linewidth=2, color='coral', label='With Baseline')
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('Gradient Norm')
axes[1, 0].set_title('Gradient Norm (越小越稳定)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 右下：Value Loss
axes[1, 1].plot(metrics_baseline["value_losses"], alpha=0.3, linewidth=0.5, color='coral')
axes[1, 1].plot(range(window-1, N_EPISODES), smooth(metrics_baseline["value_losses"], window),
                linewidth=2, color='coral')
axes[1, 1].set_xlabel('Episode')
axes[1, 1].set_ylabel('Value Loss (MSE)')
axes[1, 1].set_title('Value Network Loss (Baseline only)')
axes[1, 1].grid(True, alpha=0.3)

fig.suptitle('REINFORCE: Vanilla vs With Baseline', fontsize=14)
fig.tight_layout()
filepath = os.path.join(FIG_DIR, "13_baseline_comparison.png")
fig.savefig(filepath)
plt.close(fig)
print(f"Figure saved: {filepath}")

# 统计对比
vanilla_returns = np.array(metrics_vanilla["returns"])
baseline_returns = np.array(metrics_baseline["returns"])
last_n = 50
print(f"\\n===== 性能统计 (最后 {last_n} episodes) =====")
print(f"{'Metric':<30} {'Vanilla':<20} {'With Baseline':<20}")
print("-" * 70)
print(f"{'Mean Return':<30} {vanilla_returns[-last_n:].mean():<20.2f} {baseline_returns[-last_n:].mean():<20.2f}")
print(f"{'Std Return':<30} {vanilla_returns[-last_n:].std():<20.2f} {baseline_returns[-last_n:].std():<20.2f}")
print(f"{'Max Return':<30} {vanilla_returns.max():<20.2f} {baseline_returns.max():<20.2f}")


In [ ]:
# Cell 12: 梯度方差分析

%matplotlib inline

last_n = 50
var_vanilla = np.var(metrics_vanilla["grad_norms"][-last_n:])
var_baseline = np.var(metrics_baseline["grad_norms"][-last_n:])
ret_var_vanilla = np.var(metrics_vanilla["returns"][-last_n:])
ret_var_baseline = np.var(metrics_baseline["returns"][-last_n:])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Vanilla REINFORCE', 'REINFORCE + Baseline'], [var_vanilla, var_baseline],
            color=['steelblue', 'coral'], alpha=0.8)
axes[0].set_ylabel('Gradient Norm Variance')
axes[0].set_title('Gradient Variance (越低越稳定)')
axes[0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate([var_vanilla, var_baseline]):
    axes[0].text(i, v + 0.01, f'{v:.4f}', ha='center', fontsize=11)

axes[1].bar(['Vanilla REINFORCE', 'REINFORCE + Baseline'], [ret_var_vanilla, ret_var_baseline],
            color=['steelblue', 'coral'], alpha=0.8)
axes[1].set_ylabel('Return Variance')
axes[1].set_title('Return Variance (越低越稳定)')
axes[1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate([ret_var_vanilla, ret_var_baseline]):
    axes[1].text(i, v + 2, f'{v:.2f}', ha='center', fontsize=11)

fig.suptitle('方差对比：Vanilla vs With Baseline', fontsize=14)
fig.tight_layout()
filepath = os.path.join(FIG_DIR, "13_variance_comparison.png")
fig.savefig(filepath)
plt.close(fig)
print(f"Figure saved: {filepath}")

print(f"\\n===== 方差分析 =====")
print(f"{'Metric':<30} {'Vanilla':<16} {'With Baseline':<16} {'Reduction':<12}")
print("-" * 74)
ret_var_reduction = (ret_var_vanilla - ret_var_baseline) / ret_var_vanilla * 100
grad_var_reduction = (var_vanilla - var_baseline) / var_vanilla * 100
print(f"{'Return Variance':<30} {ret_var_vanilla:<16.2f} {ret_var_baseline:<16.2f} {ret_var_reduction:<11.1f}%")
print(f"{'Grad Norm Variance':<30} {var_vanilla:<16.4f} {var_baseline:<16.4f} {grad_var_reduction:<11.1f}%")


## 5. 如何选择好的 Baseline？

### Baseline 的理论性质

**任何与动作无关的函数都可以作为 baseline**，但效果不同：

| Baseline 选择 | 期望 | 方差降低效果 | 实现复杂度 |
|---------------|------|-------------|-----------|
| 常数 c | 无偏 | 微弱（仅中心化） | 极简 |
| 随机 baseline | 无偏 | 依赖相关性 | 视情况 |
| V(s) (最优) | 无偏 | **最大** | 需要学习 |
| 学习到的 V_phi(s) | 近似无偏 | 接近最优 | 中等 |

### 为什么 V(s) 是近乎最优的 baseline？

$V(s) = \mathbb{E}_{a \sim \pi}[Q(s,a)]$ 可以解释为："在这个状态下，平均能获得多少回报。"

- **优势**：$A(s,a) = Q(s,a) - V(s)$ 直接告诉我们这个动作相对于平均水平的好坏
- 当 $A(s,a) > 0$，增加该动作概率；$A(s,a) < 0$，降低该动作概率

### 实际考虑

在实践中，我们使用一个可学习的价值网络 $V_\phi(s)$ 作为 baseline：

1. $V_\phi(s)$ 通过 MSE 损失拟合 MC 回报：$\mathcal{L}(\phi) = \mathbb{E}[(G_t - V_\phi(s_t))^2]$
2. 学习的 $V_\phi$ 不完美，但方差降低效果仍然显著
3. 训练过程中，value network 和 policy network 可以共享特征提取器

## 6. 从 Baseline 到 Actor-Critic

### 关键洞察

当 baseline 是**可学习的价值网络** $V_\phi(s)$，并且我们使用**自举 (bootstrapping)** 代替 MC 回报时，我们就得到了 **Actor-Critic**：

| 算法 | 更新信号 | 偏差 | 方差 |
|------|---------|------|------|
| REINFORCE (无 baseline) | $G_t$ | **无偏** | 高 |
| REINFORCE + baseline | $G_t - V(s_t)$ | 无偏 | 中 |
| Actor-Critic (1-step) | $r_t + \gamma V(s_{t+1}) - V(s_t)$ | **有偏** | **低** |
| A2C (n-step) | $R_t^{(n)} - V(s_t)$ | 可控偏差 | 中低 |

### 偏差-方差权衡

| 方法 | 偏差 | 方差 |
|------|------|------|
| MC (REINFORCE) | 无偏 | 高 |
| TD(0) (1-step AC) | 有偏（估计的价值函数） | 低 |
| TD(lambda) / n-step (A2C) | 介于之间 | 介于之间 |

### 从 Baseline 到 Actor-Critic 的演进

```
REINFORCE:        loss = -log pi * G_t
REINFORCE+BL:    loss = -log pi * (G_t - V(s_t)) + MSE(V(s_t), G_t)
Actor-Critic:    loss = -log pi * (r + gamma V(s') - V(s)) + MSE(V(s), r + gamma V(s'))
A2C:             loss = -log pi * (R_t^{(n)} - V(s_t)) + MSE(V(s), R_t^{(n)}) - alpha H(pi)
```

我们将在下一节（Actor-Critic）和第 15 节（A2C）中详细讨论这些变体。

## 7. 总结

### 核心要点

1. **REINFORCE 高方差的原因**：$G_t$ 累积了整条轨迹的随机性，导致梯度估计不稳定

2. **Baseline 的核心思想**：从回报中减去一个与动作无关的基线函数，期望不变但方差减小

3. **数学证明**：$\mathbb{E}[\nabla \log \pi \cdot b(s)] = 0$，因为 $\sum_a \nabla \pi(a|s) = \nabla 1 = 0$

4. **Advantage 函数**：$A(s,a) = Q(s,a) - V(s)$ 衡量动作相对于平均水平的优势

5. **数值验证**：带 baseline 的 REINFORCE 梯度方差显著降低，训练更稳定

6. **最优 baseline**：理论上 $V(s)$（或价值加权平均）是最优选择

7. **通往 Actor-Critic**：把 MC 回报换成自举 (bootstrap) 估计，就得到了 Actor-Critic 家族

### 关键公式

| 公式 | 含义 |
|------|------|
| $\nabla J(\theta) = \mathbb{E}[\nabla \log \pi \cdot G_t]$ | REINFORCE 梯度 |
| $\nabla J(\theta) = \mathbb{E}[\nabla \log \pi \cdot (G_t - b(s))]$ | 带 baseline 的梯度 |
| $\mathbb{E}[\nabla \log \pi \cdot b(s)] = 0$ | Baseline 无偏性证明 |
| $A_t = G_t - V(s_t)$ | 优势函数估计 |
| $\mathcal{L}_{value} = \text{MSE}(V(s), G_t)$ | 价值网络损失 |

## 8. 练习

### 基础练习
1. 尝试不同的 baseline 选择：常数 baseline (如 0)，随机 baseline，观察方差变化
2. 比较不同学习率下 baseline 的效果，观察 lr 较大时 baseline 是否更有帮助
3. 观察 value network 的 loss 曲线，判断价值网络是否收敛

### 进阶练习
4. **理论**：证明最优 baseline 公式 $b^*(s) = \frac{\mathbb{E}[(\nabla \log \pi)^2 Q(s,a)]}{\mathbb{E}[(\nabla \log \pi)^2]}$
5. **实现**：在 REINFORCE with baseline 中，尝试不 detach advantages，观察训练是否会发散
6. **实现**：比较两个独立网络 vs 共享特征提取器的效果差异
7. **实验**：在 MountainCar-v0 或 Acrobot-v1 上重复实验，观察 baseline 的效果

### 思考题
8. 为什么我们说 "V(s) 是最优 baseline" 时加了引号？什么情况下 V(s) 不是最优？
9. 当价值网络 $V_\phi$ 估计不准确时，带 baseline 的 REINFORCE 还是无偏的吗？
10. 如果把 baseline 设为 Q(s,a)，会发生什么？提示：$A(s,a) = Q(s,a) - Q(s,a) = 0$

---

*下一节：[14_actor_critic.ipynb](14_actor_critic.ipynb)*